# grasp_verify: did we actually pick the piece up? verify and retry

Library notebook for `gambit_arm`. The README targets 55-60 % pick-and-place success on the
first attempt and 95 % after automatic verify-and-retry. This notebook defines the checks
that decide "the piece is in the gripper" / "the piece landed on its square", and the retry
loop that uses them. Measuring the rates is run_pick_place_trials.ipynb.

Milestone: *Hardware and execution* (gripper fingers shaped to the pieces, with grasp
verification; 95 % after verify-and-retry).

**TODO**
- [x] grasp signals data class and a rule-based verdict (gripper position + current)
- [x] retry loop with max attempts and a deterministic search pattern of small offsets
- [ ] camera check: piece gone from the source square after lift, present and centered on
      the target after place (needs gambit_vision board reading; see `camera_check`)
- [ ] calibrate thresholds per piece type on the real arm (safety.yaml `gripper` section)
- [ ] knocked-neighbor detection after a failed grasp (camera diff of the 8 neighbors)
- [ ] log every attempt (signals, verdict, offset) for the trials notebook

In [ ]:
from dataclasses import dataclass, field
import numpy as np

%run ./pick_place.ipynb

GRIP = SAFETY["gripper"]

## Signals and verdict

After commanding `gripper_closed`, a gripper with a piece between the jaws **stalls early**:
its measured position stays well above the commanded closed value, and it draws a holding
current. An empty gripper closes fully (position below `closed_empty_threshold`) and draws
little current. Any one signal can lie (a piece caught on one fingertip still stalls), so the
camera check is the final word when available.

In [ ]:
@dataclass
class GraspSignals:
    gripper_pos: float              # rad, measured after closing
    gripper_current_ma: float = None
    camera_says_gone: bool = None   # source square empty after lift (None = not checked)

@dataclass
class Verdict:
    ok: bool
    reasons: list = field(default_factory=list)

def verify_grasp(sig, cfg=GRIP):
    reasons = []
    if sig.gripper_pos < cfg["closed_empty_threshold"]:
        reasons.append(f"gripper closed to {sig.gripper_pos:.3f} rad: nothing between the jaws")
    if sig.gripper_current_ma is not None:
        lo, hi = cfg["grasp_current_ma"]
        if not lo <= sig.gripper_current_ma <= hi:
            reasons.append(f"gripper current {sig.gripper_current_ma:.0f} mA outside [{lo}, {hi}]")
    if sig.camera_says_gone is False:
        reasons.append("camera: piece still on the source square")
    return Verdict(ok=not reasons, reasons=reasons)

def camera_check(square, expect_occupied):
    """TODO: read the board with gambit_vision (classifier on the square crop) and return True
    if the square's occupancy matches `expect_occupied`; for placement also check the piece
    center is within tolerance of the square center (square_map.offset_from_center)."""
    raise NotImplementedError("camera_check needs gambit_vision board reading")

## Retry policy

On a failed grasp: open, lift back to hover, and try again at a slightly shifted grasp point
(the piece may be off-center on its square). Offsets follow a fixed pattern (center, then
+-x, +-y at `offset_step`) so trials are reproducible. After `max_attempts` give up and ask
the human; never keep poking a piece that may already be knocked over.

In [ ]:
@dataclass
class RetryPolicy:
    max_attempts: int = 3
    offset_step: float = 0.003      # m. tune on the real arm
    pattern: tuple = ((0, 0), (1, 0), (-1, 0), (0, 1), (0, -1))

    def offset(self, attempt):
        dx, dy = self.pattern[attempt % len(self.pattern)]
        return np.array([dx, dy, 0.0]) * self.offset_step

@dataclass
class AttemptLog:
    attempt: int
    offset: np.ndarray
    verdict: Verdict

def pick_with_retry(backend, square, kind, policy=RetryPolicy(), sense=None):
    """Pick `kind` from `square`, verifying after each attempt.
    backend: a pick_place Backend. sense(backend) -> GraspSignals (defaults to the gripper
    position only). Returns (success, [AttemptLog])."""
    sense = sense or (lambda be: GraspSignals(gripper_pos=be.gripper_position()))
    logs = []
    for attempt in range(policy.max_attempts):
        off = policy.offset(attempt)
        wps = pick(square, kind)
        for w in wps[1:3]:                       # shift descend + grasp, not the hover
            w.xyz = w.xyz + off
        backend.execute(plan_joint_trajectory(wps))
        verdict = verify_grasp(sense(backend))
        logs.append(AttemptLog(attempt, off, verdict))
        if verdict.ok:
            return True, logs
        # failed: open and back off to hover before the next try
        backend.execute(plan_joint_trajectory(move_to_hover(square)))
    return False, logs

def success_rates(trials):
    """trials: list of lists of AttemptLog (one list per pick). Returns
    (first-attempt rate, after-retry rate) for the README targets."""
    n = len(trials)
    first = sum(1 for t in trials if t and t[0].verdict.ok) / n
    final = sum(1 for t in trials if t and t[-1].verdict.ok) / n
    return first, final

## Self-check (fake backend: empty gripper for the first k attempts)

In [ ]:
@dataclass
class FlakyBackend(DryRunBackend):
    fail_first: int = 1
    closes: int = 0

    def gripper_position(self):
        if self.gripper == ARM["gripper_closed"]:
            # piece present -> stalls at ~0.4 rad; empty -> closes fully
            return 0.03 if self.closes <= self.fail_first else 0.4
        return self.gripper

    def execute(self, trajectory):
        if any(t.name == "grasp" for t in trajectory):
            self.closes += 1
        return super().execute(trajectory)

assert verify_grasp(GraspSignals(0.4, 200)).ok
assert not verify_grasp(GraspSignals(0.02)).ok
assert not verify_grasp(GraspSignals(0.4, 5000)).ok
assert not verify_grasp(GraspSignals(0.4, camera_says_gone=False)).ok
ok, logs = pick_with_retry(FlakyBackend(fail_first=1), "e2", "P")
assert ok and len(logs) == 2 and not logs[0].verdict.ok and np.allclose(logs[1].offset, [0.003, 0, 0])
ok, logs = pick_with_retry(FlakyBackend(fail_first=5), "e2", "P", RetryPolicy(max_attempts=3))
assert not ok and len(logs) == 3
_t = [[AttemptLog(0, 0, Verdict(True))], [AttemptLog(0, 0, Verdict(False)), AttemptLog(1, 0, Verdict(True))]]
assert success_rates(_t) == (0.5, 1.0)
print("grasp_verify OK")